# 11 Lab — The Trade Tracker

A scaffold for the capstone: build a **week-1 paper portfolio**, `summarize` every position, and
produce one **compact portfolio report** — a DataFrame of per-position metrics plus the **combined
book greeks**. Extend this into your live tracker for the 30-day program.

The week-1 book (on DEMO, spot 100, IV ~25%) deliberately mixes the required variety: an **income**
structure, a **directional** trade, and a **time spread**. Runs offline, top-to-bottom.

In [ ]:
import numpy as np
import pandas as pd
from optionslab import strategies, analyzer, greeks
SPOT, VOL = 100.0, 0.26

## 1. Build the week-1 portfolio

One income / neutral (iron condor), one directional (bull call debit spread), one time spread
(calendar). Each would have been sized to 1-2% max loss and passed the entry checklist.

In [ ]:
book = {
    "income": strategies.iron_condor((87.5,0.37),(92.5,1.01),(107.5,1.19),(112.5,0.43), expiry=45/365),
    "directional": strategies.bull_call_spread((100,3.91),(110,0.73), expiry=45/365),
    "time_spread": strategies.calendar_spread("call", 100, front_expiry=21/365, front_premium=2.66,
                                              back_expiry=45/365, back_premium=3.91),
}
for tag, pos in book.items():
    print(f"{tag:12} {pos.label}")

## 2. Summarize each position

`analyzer.summarize` returns net premium, breakevens, max P/L, POP, expected move, greeks, and DTE.
Collect the fields into one tidy per-position row.

In [ ]:
def row(tag, pos, spot, vol):
    s = analyzer.summarize(pos, spot, vol)
    g = s["greeks"]
    return {
        "tag": tag, "label": s["label"][:26],
        "net$": round(s["net_premium"], 0), "maxP$": round(s["max_profit"], 0),
        "maxL$": round(s["max_loss"], 0), "POP": round(s["probability_of_profit"], 2),
        "DTE": s["days_to_expiry"],
        "delta": round(g.delta, 1), "theta": round(g.theta, 1), "vega": round(g.vega, 1),
    }

In [ ]:
report = pd.DataFrame([row(t, p, SPOT, VOL) for t, p in book.items()])
report

## 3. Combined book greeks and totals

Sum the dollar greeks across the book (they are addable) and total the risk columns for a one-glance
portfolio picture.

In [ ]:
total_g = greeks.Greeks(0, 0, 0, 0, 0)
for pos in book.values():
    total_g = total_g + greeks.position_greeks(pos, SPOT, VOL)
print("BOOK delta/theta/vega:",
      round(total_g.delta, 1), round(total_g.theta, 1), round(total_g.vega, 1))
print("BOOK max loss if all hit worst case $:", round(report["maxL$"].sum(), 0))

The combined greeks tell you the book's real posture (here: net long delta from the
directional spread, and a theta/vega blend from the condor and calendar). Compare `delta` and `vega`
against the portfolio limits in your trading plan before adding anything new.

In [ ]:
LIMITS = {"delta": (-200, 200), "vega": (-50, 50)}
for k in ("delta", "vega"):
    v = getattr(total_g, k); lo, hi = LIMITS[k]
    print(f"net {k} {round(v,1):>7}  limit [{lo}, {hi}]  ->", "OK" if lo <= v <= hi else "BREACH")

## 4. A compact printable report

One block you could paste into the daily journal: per-position risk + the book totals.

In [ ]:
print(report.to_string(index=False))
print("-" * 60)
print(f"BOOK  net$={report['net$'].sum():.0f}  maxL$={report['maxL$'].sum():.0f}  "
      f"delta={total_g.delta:.1f}  theta={total_g.theta:.1f}  vega={total_g.vega:.1f}")

## Experiments

1. Add a fourth position (a cash-secured put or a diagonal) and re-run — watch the book greeks and
   total max loss update. Does it still fit your limits?
2. Add a **days_forward** column: for each position, compute `payoff.pnl_at(pos, SPOT, 10/365, VOL)`
   to see mark-to-model P&L 10 days out with no price move (theta at work).
3. Replace the directional leg with a **bearish** one and confirm the book's net delta flips sign.
4. Track the same book over a price move: set `SPOT = 96` and re-run — which position's greeks change
   most, and which is now tested?
5. Turn `row()` into your live tracker: add columns for entry date, profit target, and a "status vs
   plan" flag, and append a row each time you open a paper trade in the 30-day program.